# Simran Amesar - 100007050

In [466]:
import tensorflow as tf
from tensorflow.keras import layers 
from tensorflow.keras.preprocessing.text import Tokenizer 
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [467]:
texts = [
 
    "I love this movie",
    "This film is amazing",
    "Very good acting",
    "Excellent story",
    "I enjoyed the movie",
    "I hate this movie",
    "Terrible film",
    "Very boring story",
    "Worst acting ever",
    "I disliked the movie"
 
]
labels = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

In [468]:
vocab_size = 1000
max_length = 8

In [469]:
tokenizer = Tokenizer()
tokenizer = Tokenizer(num_words = vocab_size, oov_token = "<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen = max_length, padding = "post")

print("Word Index:")
print(tokenizer.word_index)



Word Index:
{'<OOV>': 1, 'i': 2, 'movie': 3, 'this': 4, 'film': 5, 'very': 6, 'acting': 7, 'story': 8, 'the': 9, 'love': 10, 'is': 11, 'amazing': 12, 'good': 13, 'excellent': 14, 'enjoyed': 15, 'hate': 16, 'terrible': 17, 'boring': 18, 'worst': 19, 'ever': 20, 'disliked': 21}


In [470]:
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()

        
        """ 1. Token Embedding
            Converts each word's integer ID into a 16-dimensional vector.
            The vectors are learned during training.

        """
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
        

        """ 2. Positional Embedding
            Gives the model a sense of word order.
            Transformers process all words at once, not sequentially like RNNs.
            Generates position indices, then adds position vectors on top of token vectors.
        """
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )
 
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb

In [471]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()


        """ 3. Multi-Head Attention
                Lets every word look at every other word in the sequence and decide how much to "pay attention" to each one.
                num_heads=2 means it does this from 2 different perspectives simultaneously, then combines the results.
        """
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )


        """ 4. Feed Forward Network
                Processes each token's representation independently through 2 Dense layers.
                First layer expands from 16 - 32 dims.
                Second layer compresses back from 32 - 16 dims.
                Attention finds relationships between words,

        """
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
 
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
 
    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence
        attention_output = self.attention(inputs, inputs)


        """ 5. Add & Normalize
                First Add & Norm — after attention: + adds the original input back to the attention output. This prevents vanishing gradients and helps the model
                retain information.
                Second Add & Norm — after FFN: preserves what was learned before FFN, then normalize again to keep values going to the next layer.
        """
        out1 = self.layernorm1(inputs + attention_output)
        # Feed-forward network
        ffn_output = self.ffn(out1)
        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)
 
        return out2

In [472]:
# build the model
embed_dim = 16
num_heads = 2
ff_dim = 32
inputs = layers.Input(shape=(max_length,))
 
x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)
 
x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)
 

""" 6. Output Layer
        GlobalAveragePooling1D collapses the sequence dimension, the entire sentence into one fixed-size representation.
        The model needs a single scalar prediction per sentence, not 20 separate vectors. Pooling summarises the whole sequence; 
        sigmoid converts the score to a probability.

"""
x = layers.GlobalAveragePooling1D()(x)
 
outputs = layers.Dense(1, activation="sigmoid")(x)
 
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [473]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]

)

In [474]:
model.summary()

Model: "functional_89"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_88 (InputLayer)     │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_44 │ (None, 8, 16)          │        16,128 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_44            │ (None, 8, 16)          │         3,296 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_44     │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_134 (Dense)               │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,441 (75.94 KB)

 Trainable params: 19,441 (75.94 KB)

 Non-trainable params: 0 (0.00 B)

In [475]:
model.fit(
    X,
    labels,
    epochs=30, 
    batch_size= 2,
    verbose=1
)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5000 - loss: 1.0333
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6000 - loss: 0.8167    
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5000 - loss: 0.7317
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6000 - loss: 0.6941    
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7000 - loss: 0.6179
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7000 - loss: 0.6167
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7000 - loss: 0.5881
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7000 - loss: 0.6042
Epoch 9/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8000 - loss: 0.5657
Epoch 10/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8000 - loss: 0.5568    
Epoch 11/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9000 - loss: 0.5102
Epoch 12/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.4908
E

In [476]:
loss, accuracy = model.evaluate(X, labels)

print("Accuracy:", accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 1.0000 - loss: 0.1907
Accuracy: 1.0


In [477]:
test_sentences = [
    "I love the film",
    "This movie was awful",
    "This movie is excellent", 
    "I would watch it again.",
    "I like it", 
    "Don't recommend"

]
 
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)
 
for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive \n")
    else:
        print("Prediction: Negative \n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
I love the film -> 0.7127735
Prediction: Positive 

This movie was awful -> 0.82665896
Prediction: Positive 

This movie is excellent -> 0.92989284
Prediction: Positive 

I would watch it again. -> 0.7867148
Prediction: Positive 

I like it -> 0.58770496
Prediction: Positive 

Don't recommend -> 0.59602475
Prediction: Positive 



# Part 1 - Architecture

## Task 1 <br>
### Locate the following components in the code:
- Token Embedding 
- Positional Embedding 
- Multi-Head Attention 
- Feed Forward Network 
- Add & Normalize 
- Output Layer 

# Task 2 <br>
## Create a diagram showing the flow of data through the model.

# Task 3 
### For each component explain:
- What does it do ?
- Why is it needed?
- What might happen if it is removed?

| Component | What does it do? | Why is it needed? | What if removed? |
|---|---|---|---|
| **Token Embedding** | Converts each word ID into a 16-dim learned vector | Neural networks can't process raw integers; vectors let the model learn that similar words are close together | No numerical representation of words — training cannot begin |
| **Positional Embedding** | Adds a learned 16-dim vector for each position  | Transformers process all tokens at once; without this, word order is invisible | 'I love this' and 'this love I' are identical to the model |
| **Multi-Head Attention** | Every word computes a weighted sum over all other words (how much to attend to each) | Meaning depends on context; attention links related words across positions  | Each word is processed in isolation — no relationships between words can be learned |
| **Add & Normalize** | Residual skip connection adds input back to output; LayerNorm rescales to mean≈0, std≈1 | Prevents vanishing/exploding gradients; keeps training numerically stable so the network converges | Training becomes unstable or fails entirely — loss may diverge or gradients vanish |
| **Feed-Forward Network** | Two Dense layers per token: expands 16→32 (ReLU) then compresses 32→16 | Attention is linear; FFN adds non-linearity so the model can learn complex, abstract patterns | Model becomes purely linear — cannot learn non-linear decision boundaries, severely limiting capability |
| **Output Layer** | Pooling averages 8 token vectors into one; Dense(1, sigmoid) maps to probability | Need a single scalar prediction per sentence; sigmoid + binary crossentropy is a matched pair | Without pooling: shape mismatch crashes the model. Without sigmoid: output is unbounded and loss breaks mathematically |

---

## Part 2 - Predict before Running
Before teh training model:
Predict:
1. Will the model achieve 50%, 70%, 90% or 100% accuracy? 
 - Answer: The model will likely reach 100% accuracy on the training set.
           With only 10 samples and 30 epochs, the transformer has far more parameters than data points, so it will simply memorise every example.It is overfitting.
2. Which words do you think will learn as strong positive indicators?
 - Answer: Words that appear exclusively in positive sentences should get high positive weights. The model has only 10 sentences, so it learns word–label co-occurrence  almost perfectly.
3. Which words do you think will learn as strong negative indicators?
 - Answer: Words that appear exclusively in negative sentences should push the output toward 0.


## Postive Indicators:
    - love, amazing, good, excellent, enjoyed
    - They appear only in positive (label=1) sentences 

## Negative Indicators:
    - hate, terrible, boring, worst, disliked 
    - They appear only in negative (label=0) sentences

## Run the model afterwards and compare your predictions with the actual reuslts.
- After running the model we see 100% training accuracy from epoch 12 onward, confirming the memorisation prediction.
The words predicted as positive/negative indicators align with the model's learned behaviour, sentences containing only positive indicator words score > 0.5 
However, 'This movie was awful' is predicted **Positive** in the test output, showing the model cannot generalise: 'awful' was never in the training set so it maps to <OOV> that is Out of vocabulary and the model defaults to its training bias.


# Part 4 - Hyperparameter Challenge 

## You have limited computing resoureses You must recommend a configuation.

In [478]:
import time

configs = [
    {"num_heads": 1, "embed_dim": 8,  "ff_dim": 16},
    {"num_heads": 2, "embed_dim": 16, "ff_dim": 32},
    {"num_heads": 4, "embed_dim": 32, "ff_dim": 64},
    {"num_heads": 8, "embed_dim": 64, "ff_dim": 128},
]

results = []

for cfg in configs:
    nh, ed, fd = cfg["num_heads"], cfg["embed_dim"], cfg["ff_dim"]
    inp = layers.Input(shape=(max_length,))
    x = TokenAndPositionEmbedding(max_length=max_length, vocab_size=vocab_size, embed_dim=ed)(inp)
    x = TransformerBlock(embed_dim=ed, num_heads=nh, ff_dim=fd)(x)
    x = layers.GlobalAveragePooling1D()(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m = tf.keras.Model(inputs=inp, outputs=out)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    t0 = time.time()
    m.fit(X, labels, epochs=30, batch_size=2, verbose=0)
    elapsed = time.time() - t0

    _, acc = m.evaluate(X, labels, verbose=0)
    params = m.count_params()
    results.append((nh, ed, fd, params, round(acc * 100), round(elapsed, 2)))
    print(f"heads={nh}, embed_dim={ed}: accuracy={acc*100:.0f}%, time={elapsed:.2f}s, params={params:,}")

heads=1, embed_dim=8: accuracy=100%, time=1.92s, params=8,673
heads=2, embed_dim=16: accuracy=100%, time=1.84s, params=19,441
heads=4, embed_dim=32: accuracy=100%, time=1.87s, params=53,409
heads=8, embed_dim=64: accuracy=100%, time=2.05s, params=214,081


| Heads | Embedding Dim | FF Dim | Parameters | Accuracy | Training Time |
|:---:|:---:|:---:|:---:|:---:|---|
| 1 | 8 | 16 | ~8 K | ~100% | 1.92s |
| 2 | 16 | 32 | ~19 K | 100% | 1.84s | 
| 4 | 32 | 64 | ~53 K | 100% | 1.87s | 
| 8 | 64 | 128 | ~214 K | 100% | 2.05s | 